# NB4 — Selected Model, Saved Artifacts, and App Presentation


This is an English, curated retrospective of the first bootcamp project. It follows the original presentation and retains the experiments needed to understand the selected model. It does not include the later Claude Code/GitHub upgrade.

**Two kinds of evidence:** snapshot statistics are computed from the supplied CSV; historical model metrics are transcribed from the original notebooks' stored outputs and explicitly labeled. The models have not been retrained for this edition. Set the relevant run flag to opt into an adapted historical rerun; scores may differ because the original `df_keyword.csv`, environment, and all intermediate states are unavailable. The supplied `CleanedDataset.csv` is used as a documented substitute, not asserted to be byte-identical to that intermediate file.

Cells carry original filenames and zero-based cell references in metadata. No API call or model download runs by default. The original evaluation limitations are noted without silently replacing the first project with an improved pipeline.


In [1]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

# Place the supplied CSV files in data/, or set RASFF_DATA_DIR.
DATA_DIR = Path(os.environ.get("RASFF_DATA_DIR", "data"))
clean_path = DATA_DIR / "CleanedDataset.csv"
if not clean_path.exists():
    raise FileNotFoundError("Place the original CleanedDataset.csv in data/ or set RASFF_DATA_DIR.")
df = pd.read_csv(clean_path).reset_index(drop=True)
RUN_EXPERIMENTS = False
RUN_LLM_SAMPLE = False
SAVE_ARTIFACTS = False
print(f"Provided clean snapshot: {df.shape[0]:,} rows x {df.shape[1]} columns")


Provided clean snapshot: 27,397 rows x 23 columns


## 1. What “best model” meant in the first project
The selected presentation result was not the highest test accuracy observed anywhere in the notebooks. The exploration included training/test comparisons, regularization, rare-category grouping, target-encoding smoothing, and separate PCA branches.

| Historical candidate | Training accuracy | Test accuracy | Interpretation |
|---|---:|---:|---|
| Deeper-tree candidate, original cell 6 | 0.9737 | 0.8775 | Different tree settings on the earlier full representation |
| Separate PCA branch, original cell 10 | 0.9593 | 0.8710 | Different feature representation; not a controlled parameter-only comparison |
| Selected candidate, original cell 4 | 0.9332 | 0.8736 | Exact result corresponding to the final presentation |

The lower training score and smaller observed gap help explain the selection narrative; these records do not prove that regularization alone caused the difference. “Best” here means the candidate selected for that first presentation, not a independently validated production model.


## 2. Exact selected input path
Original `08.bestmodelSelection.ipynb`, cells **2 → 3 → 4 → 5**:

| Raw input | Transformation |
|---|---|
| subject | all-MiniLM-L6-v2 embedding, 384 dimensions |
| notifying_country, origin, simplified_hazard | Rare grouping (<50), target encoding with smoothing=10 |
| classification, Hazard_Type | One-hot, drop first |
| category | Rare grouping (<50) into category_grouped, then one-hot |

The stored matrix is **27,397 × 427**. The original code computed PCA(100) but concatenated the original embedding matrix into `X`; the unused PCA calculation is removed here. This selected model is not a 100-dimensional PCA model.

**Historical implementation note:** target encoding used all labels before train/test splitting. The next cell intentionally retains that order to document the first model; it is not a recommended validation pipeline. A later improvement notebook should split first and use training-only/out-of-fold target encoding, plus an explicit prediction-time feature review.


In [2]:
recorded_final = pd.Series({
    "Evidence source": "08.bestmodelSelection.ipynb, cell 4 (stored output)",
    "Numerical features": 427,
    "Training rows": 19177, "Test rows": 8220,
    "Split": "Random stratified 70/30; seed 42",
    "Training accuracy": .9332012306408719,
    "Test accuracy": .8736009732360097,
    "Macro F1 (rounded)": .87,
    "Serious precision (rounded)": .86,
    "Serious recall (rounded)": .91,
    "Other-label support": 3793, "Serious support": 4427,
})
display(recorded_final.to_frame("Recorded selected-model result"))


,Recorded selected-model result
Evidence source,"08.bestmodelSelection.ipynb, cell 4 (stored ou..."
Numerical features,427
Training rows,19177
Test rows,8220
Split,Random stratified 70/30; seed 42
Training accuracy,0.933201
Test accuracy,0.873601
Macro F1 (rounded),0.87
Serious precision (rounded),0.86
Serious recall (rounded),0.91


### Optional historical preprocessing reconstruction
Only execute to inspect the historical path. All-MiniLM requires a model download; the full embeddings take time. The supplied clean snapshot substitutes for the unavailable original intermediate CSV. No claim of exact score reproduction is made.


In [3]:
if RUN_EXPERIMENTS:
    df = df.copy().reset_index(drop=True)
    import pandas as pd
    import numpy as np

    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.metrics import accuracy_score, classification_report
    from category_encoders import TargetEncoder
    from sentence_transformers import SentenceTransformer

    from xgboost import XGBClassifier

    # =====================================================
    # 1. Recode target (same as before)
    # =====================================================
    def recode_risk(risk):
        if risk in ['no risk', 'not serious', 'potential risk', 'undecided', 'potentially serious']:
            return 0
        elif risk == 'serious':
            return 1
        else:
            return -1

    df["risk_decision_2class"] = df["risk_decision"].apply(recode_risk)
    df = df[df["risk_decision_2class"] != -1]

    TARGET = "risk_decision_2class"

    # =====================================================
    # 2. Define feature groups
    # =====================================================
    embedding_cols = ["subject"]
    onehot_cols = ["classification", "Hazard_Type"]

    target_encoding_cols = ["notifying_country", "origin", "simplified_hazard"]

    # Fill missing
    for col in embedding_cols + onehot_cols + target_encoding_cols:
        df[col] = df[col].fillna("missing")


    # =====================================================
    # 2a. Group rare categories to reduce overfitting
    # =====================================================
    # Category: group rare categories (<50 samples)
    threshold_category = 50
    counts = df['category'].value_counts()
    rare_categories = counts[counts < threshold_category].index
    df['category_grouped'] = df['category'].replace(rare_categories, 'other')

    # Target encoding columns: group rare categories
    for col in target_encoding_cols:
        counts = df[col].value_counts()
        rare = counts[counts < 50].index
        df[col] = df[col].replace(rare, 'other')


    # =====================================================
    # 3. SentenceTransformer Embedding
    # =====================================================
    from sentence_transformers import SentenceTransformer

    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embedder = SentenceTransformer(model_name)

    def embed_column(series):
        return embedder.encode(series.tolist(), convert_to_numpy=True)

    print("Generating embeddings for:", embedding_cols)

    embedding_features = []
    for col in embedding_cols:
        emb = embed_column(df[col])
        # Rename embedding columns (e.g., subject_emb_0, subject_emb_1...)
        emb_df = pd.DataFrame(
            emb,
            columns=[f"{col}_emb_{i}" for i in range(emb.shape[1])]
        )
        embedding_features.append(emb_df)

    embedding_df = pd.concat(embedding_features, axis=1)

    # Unused PCA computation omitted from this selected path.

    # =====================================================
    # 4. Smoothed Target Encoding
    # =====================================================
    from category_encoders import TargetEncoder

    print("Applying Smoothed Target Encoding:", target_encoding_cols)

    te = TargetEncoder(cols=target_encoding_cols, smoothing=10)
    target_encoded_df = te.fit_transform(df[target_encoding_cols], df[TARGET])


    # =====================================================
    # 5. One-Hot Encoding
    # =====================================================
    print("Applying One-Hot Encoding:", onehot_cols + ['category_grouped'])

    ohe = OneHotEncoder(drop="first", sparse_output=False)
    ohe_array = ohe.fit_transform(df[onehot_cols + ['category_grouped']])

    ohe_df = pd.DataFrame(
        ohe_array,
        columns=ohe.get_feature_names_out(onehot_cols + ['category_grouped'])
    )


    # =====================================================
    # 6. Combine all engineered features
    # =====================================================
    X = pd.concat([embedding_df, target_encoded_df, ohe_df], axis=1)
    y = df[TARGET]

    print("Final feature matrix shape:", X.shape)


    # =====================================================
    # 7. Sanitize feature names for XGBoost compatibility
    # =====================================================
    X.columns = (
        X.columns
          .str.replace(r'[\[\]\{\}\:\"\'\,]', '_', regex=True)
          .str.replace(r'[^A-Za-z0-9_]+', '_', regex=True)
    )
else:
    print("Optional historical experiment skipped.")


Optional historical experiment skipped.


## 3. Split, fit, and evaluate the selected XGBoost settings
The historical model used 500 trees, maximum depth 4, learning rate 0.05, row/column subsampling 0.8, gamma 0.1, minimum child weight 1, L1 penalty 1, and L2 penalty 2. The optional rerun defaults to CPU for portability; the source requested CUDA. The obsolete GPU predictor argument is omitted. Neither edit is a claim that the historical scores were newly reproduced.


In [4]:
selected_parameters = dict(
    objective="binary:logistic", eval_metric="logloss", tree_method="hist",
    random_state=42, n_estimators=500, max_depth=4, learning_rate=.05,
    subsample=.8, colsample_bytree=.8, gamma=.1, min_child_weight=1,
    reg_alpha=1, reg_lambda=2,
)
display(pd.Series(selected_parameters).to_frame("Selected settings"))


,Selected settings
objective,binary:logistic
eval_metric,logloss
tree_method,hist
random_state,42
n_estimators,500
max_depth,4
learning_rate,0.05
subsample,0.8
colsample_bytree,0.8
gamma,0.1


In [5]:
if RUN_EXPERIMENTS:
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, classification_report
    from xgboost import XGBClassifier
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.3, random_state=42, stratify=y)
    xgb_model = XGBClassifier(**selected_parameters)
    xgb_model.fit(X_train, y_train)
    print("RERUN training accuracy:", accuracy_score(y_train, xgb_model.predict(X_train)))
    print("RERUN test accuracy:", accuracy_score(y_test, xgb_model.predict(X_test)))
    print(classification_report(y_test, xgb_model.predict(X_test)))
else:
    print("Optional historical experiment skipped.")


Optional historical experiment skipped.


## 4. Preserve the recorded classification report
The selected run's original test report is reproduced below as text evidence. It is not attached as the output of the optional rerun cell. The presentation's serious precision label of 0.91 conflicts with this report; **precision is 0.86 and recall is 0.91**.


```text
=== Test Data Performance ===
Accuracy: 0.8736009732360097
              precision    recall  f1-score   support

           0       0.89      0.83      0.86      3793
           1       0.86      0.91      0.89      4427

    accuracy                           0.87      8220
   macro avg       0.88      0.87      0.87      8220
weighted avg       0.87      0.87      0.87      8220
```


## 5. Model export: make the selected objects explicit
The source recorded saving the model, target encoder, one-hot encoder, PCA object, and preprocessing configuration. The actual pickle files were not supplied. This edition omits the unused PCA object and offers an optional local export only after a rerun.

The feature list and column order belong with the model. This export documents the historical candidate; it does not certify that the first app used the same objects.


In [6]:
if RUN_EXPERIMENTS and SAVE_ARTIFACTS:
    import json
    import joblib
    artifact_dir = Path("artifacts/historical_candidate")
    artifact_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(xgb_model, artifact_dir / "xgb_model.pkl")
    joblib.dump(te, artifact_dir / "target_encoder.pkl")
    joblib.dump(ohe, artifact_dir / "onehot_encoder.pkl")
    config = {"subject_embedding_model": model_name,
              "target_encoding_fields": target_encoding_cols,
              "onehot_fields": onehot_cols + ["category_grouped"],
              "rare_threshold": 50, "feature_order": X.columns.tolist(),
              "historical_warning": "Target encoding fitted before splitting; not independently validated."}
    (artifact_dir / "preprocessing_config.json").write_text(json.dumps(config, indent=2))
    print("Historical rerun artifacts saved locally.")
else:
    print("No model artifacts written; a rerun and SAVE_ARTIFACTS=True are required.")


No model artifacts written; a rerun and SAVE_ARTIFACTS=True are required.


## 6. Dashboard export and the original app presentation
`09.Dashboard.ipynb` prepared a 13-column dashboard CSV. The presentation showed a simple Streamlit interface with subject, hazard type, and origin inputs and numerical risk displays. The original Streamlit `.py` file and model pickle files are not in the supplied materials.

Therefore this notebook retains the actual dashboard-data step and documents the demonstrated interface, without inventing an original `predict()` implementation or asserting that the interface was validated against the selected seven-field model. A screenshot proves what was shown, not the prediction path behind it.


In [7]:
dashboard_df = df.copy()
dashboard_df["risk_decision_2class"] = dashboard_df["risk_decision"].eq("serious").astype(int)
dashboard_columns = ["reference", "date", "year", "month", "type", "category", "origin",
                     "notifying_country", "simplified_hazard", "Hazard_Type", "classification",
                     "risk_decision", "risk_decision_2class"]
dashboard = dashboard_df[dashboard_columns]
print("Dashboard export shape:", dashboard.shape)
if SAVE_ARTIFACTS:
    export_dir = Path("artifacts")
    export_dir.mkdir(exist_ok=True)
    dashboard.to_csv(export_dir / "dashboard1_ready.csv", index=False)
    print("Dashboard CSV saved locally.")


Dashboard export shape: (27397, 13)


## 7. Where the retrospective ends
The first project delivered structured data, comparative experiments, a selected model result, artifact-saving code, and a simple app presentation. The following improvement series should examine prediction-time availability, leakage-free preprocessing, fixed temporal validation, model/app parity, and reproducibility. Those changes are deliberately not merged into this historical edition.
